In [36]:
import os, time
from utils import *
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_W, IMG_H = 280, 70
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789 "
IMG_DIR = Path("dataset/images")
CSV_PATH = Path("dataset/labels.csv")

MIN_LEN = 3
MAX_LEN = 14

BATCH_SIZE = 128
EPOCHS = 15
LR = 3e-5
WEIGHT_DECAY = 2e-2
NUM_WORKERS = 2 if os.name == "nt" else 4

WARMUP_EPOCHS = 3
MIN_LR_FACTOR = 0.05

GRAD_CLIP_NORM = 5.0
EARLY_STOP_PATIENCE = 10

CKPT_DIR = Path("checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

ALLOWED = set(ALPHABET)

def normalize_text(s: str) -> str:
    s = str(s)
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = "".join(ch for ch in s if ch in ALLOWED)
    return s

BLANK_IDX = 0
char2idx = {c: i + 1 for i, c in enumerate(ALPHABET)}
idx2char = {i + 1: c for i, c in enumerate(ALPHABET)}
NUM_CLASSES = len(ALPHABET) + 1

def encode_text(text: str) -> torch.Tensor:
    text = normalize_text(text)
    return torch.tensor([char2idx[c] for c in text], dtype=torch.long)

def ctc_greedy_decode(logits: torch.Tensor) -> list[str]:
    pred = logits.argmax(dim=-1)  # [T, B]
    pred = pred.detach().cpu().numpy()

    out = []
    for b in range(pred.shape[1]):
        seq = pred[:, b].tolist()
        collapsed = []
        prev = None
        for p in seq:
            if p != prev:
                collapsed.append(p)
            prev = p
        collapsed = [p for p in collapsed if p != BLANK_IDX]
        out.append("".join(idx2char.get(p, "") for p in collapsed))
    return out

def cer(pred: str, gt: str) -> float:
    a, b = pred, gt
    if len(b) == 0:
        return 0.0 if len(a) == 0 else 1.0
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev = dp[0]
        dp[0] = i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            cost = 0 if ca == cb else 1
            dp[j] = min(
                dp[j] + 1,      # deletion
                dp[j-1] + 1,    # insertion
                prev + cost     # substitution
            )
            prev = cur
    return dp[-1] / max(1, len(b))


In [37]:
ds = OCRDataset(IMG_DIR, CSV_PATH)
len(ds)

n = len(ds)
test_size = int(0.1 * n)
val_size = int(0.1 * n)
train_size = n - val_size - test_size

train_ds, val_ds, test_ds = random_split(
    ds, [train_size, val_size, test_size]
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn
)

model = CRNN(NUM_CLASSES).to(DEVICE)

In [38]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def get_lr(epoch: int) -> float:
    if epoch < WARMUP_EPOCHS:
        return LR * (epoch + 1) / WARMUP_EPOCHS
    t = (epoch - WARMUP_EPOCHS) / max(1, (EPOCHS - WARMUP_EPOCHS))
    cos = 0.5 * (1 + math.cos(math.pi * t))
    return LR * (MIN_LR_FACTOR + (1 - MIN_LR_FACTOR) * cos)

ctc_loss = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)

use_amp = (DEVICE == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_chars = 0
    total_cer = 0.0

    for x, y_cat, y_lens, texts in loader:
        x = x.to(DEVICE, non_blocking=True)
        y_cat = y_cat.to(DEVICE, non_blocking=True)
        y_lens = y_lens.to(DEVICE, non_blocking=True)

        logits = model(x)  # [T,B,C]
        log_probs = logits.log_softmax(dim=-1)

        T, B, C = log_probs.shape
        input_lens = torch.full((B,), T, dtype=torch.long, device=DEVICE)

        loss = ctc_loss(log_probs, y_cat, input_lens, y_lens)
        total_loss += loss.item() * B

        # CER
        preds = ctc_greedy_decode(logits)
        for p, gt in zip(preds, texts):
            total_cer += cer(p, gt) * len(gt)
            total_chars += len(gt)

    return total_loss / len(loader.dataset), (total_cer / max(1, total_chars))

def train_one_epoch(model, loader, epoch):
    model.train()
    total_loss = 0.0

    # set LR for this epoch
    lr = get_lr(epoch)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    for x, y_cat, y_lens, _texts in loader:
        x = x.to(DEVICE, non_blocking=True)
        y_cat = y_cat.to(DEVICE, non_blocking=True)
        y_lens = y_lens.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(x)
            log_probs = logits.log_softmax(dim=-1)
            T, B, C = log_probs.shape
            input_lens = torch.full((B,), T, dtype=torch.long, device=DEVICE)
            loss = ctc_loss(log_probs, y_cat, input_lens, y_lens)

        scaler.scale(loss).backward()

        # unscale + clip
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset), lr

In [39]:
best_val_cer = 1e9
best_epoch = -1
no_improve = 0

start_time = time.time()

for epoch in range(EPOCHS):
    tr_loss, lr = train_one_epoch(model, train_loader, epoch)
    va_loss, va_cer = evaluate(model, val_loader)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | lr={lr:.6f} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_CER={va_cer:.4f}")

    improved = va_cer < best_val_cer - 1e-6
    if improved:
        best_val_cer = va_cer
        best_epoch = epoch + 1
        no_improve = 0

        torch.save({
            "model": model.state_dict(),
            "alphabet": ALPHABET,
            "img_w": IMG_W,
            "img_h": IMG_H,
            "best_val_cer": best_val_cer,
            "epoch": best_epoch,
        }, CKPT_DIR / "crnn_ctc_best.pt")
    else:
        no_improve += 1

    if no_improve >= EARLY_STOP_PATIENCE:
        print(f"Early stopping: no improvement for {EARLY_STOP_PATIENCE} epochs.")
        break

total_time = time.time() - start_time
print(f"\nBest val CER: {best_val_cer:.6f} at epoch {best_epoch}")
print(f"Total training time: {total_time/60:.2f} min")

ckpt = torch.load(CKPT_DIR / "crnn_ctc_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])

te_loss, te_cer = evaluate(model, test_loader)
print(f"TEST | loss={te_loss:.4f} | CER={te_cer:.4f}")

Epoch 01/30 | lr=0.000100 | train_loss=7.8718 | val_loss=4.2277 | val_CER=1.0000
Epoch 02/30 | lr=0.000200 | train_loss=2.5315 | val_loss=0.2185 | val_CER=0.0272
Epoch 03/30 | lr=0.000300 | train_loss=0.0546 | val_loss=0.0372 | val_CER=0.0077
Epoch 04/30 | lr=0.000300 | train_loss=0.0200 | val_loss=0.0190 | val_CER=0.0034
Epoch 05/30 | lr=0.000299 | train_loss=0.0138 | val_loss=0.0108 | val_CER=0.0023
Epoch 06/30 | lr=0.000296 | train_loss=0.0070 | val_loss=0.0557 | val_CER=0.0134
Epoch 07/30 | lr=0.000291 | train_loss=0.0071 | val_loss=0.0150 | val_CER=0.0040
Epoch 08/30 | lr=0.000285 | train_loss=0.0570 | val_loss=0.0131 | val_CER=0.0022
Epoch 09/30 | lr=0.000277 | train_loss=0.0059 | val_loss=0.0058 | val_CER=0.0013
Epoch 10/30 | lr=0.000267 | train_loss=0.0024 | val_loss=0.0037 | val_CER=0.0008
Epoch 11/30 | lr=0.000255 | train_loss=0.0015 | val_loss=0.0027 | val_CER=0.0007
Epoch 12/30 | lr=0.000243 | train_loss=0.0012 | val_loss=0.0015 | val_CER=0.0003
Epoch 13/30 | lr=0.000229 | 